> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 5 · Notebook 05 — VWAP, volume profile and higher timeframes

**Sessions:** S8 (Price-volume-time group & multi-timeframe) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Compute session VWAP that resets every day, and an anchored VWAP.
2. Find the point of control of a volume profile.
3. Put an hourly value on 5-minute bars without looking ahead.
4. Watch the default `resample` leak the future into a backtest.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()

## 1. Intraday bars

Five days of 5-minute bars, each **stamped at its close** (09:35 … 16:00 New York), with the usual U-shaped volume.

In [ ]:
bars = p.intraday_bars(days=5)
display(bars.head(3))
bars["volume"].groupby(bars.index.time).mean().plot(title="Average volume by time of day (5-minute bars)"); plt.show()

## 2. Session VWAP

`VWAP = Σ(price · volume) / Σ(volume)` from the session's first bar, **resetting each day**. Group by `bars.index.date` and take cumulative sums within each group.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def session_vwap(df):
    day = df.index.date
    pv = (df["close"] * df["volume"]).groupby(day).cumsum()
    vol = df["volume"].groupby(day).cumsum()
    return pv / vol

mine = p.attempt(session_vwap, bars)
mine = p.check("session_vwap", mine, p.session_vwap(bars))
mine.iloc[76:80]

In [ ]:
low_bar = int(np.argmin(bars["close"].to_numpy()[:200]))
avwap = p.anchored_vwap(bars["close"], bars["volume"], low_bar)
x = np.arange(len(bars))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(x, bars["close"], lw=1, color="#8a8984", label="close")
ax.plot(x, p.session_vwap(bars), label="session VWAP (resets daily)")
ax.plot(x, avwap, label=f"VWAP anchored at bar {low_bar} (the low)")
for d in range(1, 5):
    ax.axvline(d * 78, color="#e6e5e0", lw=1)
ax.set(xlabel="bar", title="Session vs anchored VWAP"); ax.legend(); plt.show()

## 3. Volume profile and the point of control

A volume profile sums volume by **price** instead of by time. Its busiest price bin is the **point of control** (POC), a level traders watch as support or resistance. Use `np.histogram(price, bins=bins, weights=volume)` and return the centre of the fullest bin.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def point_of_control(price, volume, bins=40):
    hist, edges = np.histogram(price, bins=bins, weights=volume)
    i = int(np.argmax(hist))
    return float((edges[i] + edges[i + 1]) / 2)

mine = p.attempt(point_of_control, bars["close"], bars["volume"])
mine = p.check("point_of_control", mine, p.point_of_control(bars["close"], bars["volume"]))
mine

In [ ]:
hist, edges = np.histogram(bars["close"], bins=40, weights=bars["volume"])
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh((edges[:-1] + edges[1:]) / 2, hist, height=np.diff(edges) * 0.9)
ax.axhline(mine, color=p.PALETTE[7], ls="--", label=f"POC {mine:.2f}")
ax.set(xlabel="volume", ylabel="price", title="Volume profile, 5 days"); ax.legend(); plt.show()

## 4. Higher timeframes without look-ahead

A 5-minute strategy often uses an hourly indicator. The hourly bar that covers 10:00–11:00 is **known only at 11:00**. With close-stamped bars: resample with `closed="right", label="right"` so each hourly bar is stamped when it completes, take `.last()`, drop the empty hours, and forward-fill onto the 5-minute index.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def align_higher_tf(close, rule="1h"):
    h = close.resample(rule, closed="right", label="right").last().dropna()
    return h.reindex(close.index, method="ffill")

mine = p.attempt(align_higher_tf, bars["close"])
mine = p.check("align_higher_tf", mine, p.align_higher_tf(bars["close"]))
pd.DataFrame({"close": bars["close"], "hourly (honest)": mine, "hourly (default resample)": p.align_higher_tf_leaky(bars["close"])}).iloc[4:16].round(2)

Look at the default column: before 10:00 it already shows the **09:55** close, and from 10:00 on it shows the close of **10:55**, up to fifty-five minutes early. The honest column only changes when an hour completes. Any rule that compares price to that value is trading on the future. A toy rule makes it obvious: "buy for one bar when the 5-minute close is below the hourly close".

In [ ]:
big = p.intraday_bars(days=120, seed=11)
nxt = big["close"].shift(-1) / big["close"] - 1
for name, htf in [("honest", p.align_higher_tf(big["close"])), ("default resample", p.align_higher_tf_leaky(big["close"]))]:
    sig = big["close"] < htf
    print(f"{name:17s}: mean next-bar return when signalled {nxt[sig].mean() * 1e4:+.2f} bp, otherwise {nxt[~sig].mean() * 1e4:+.2f} bp")

## Wrap-up

* VWAP resets each session; anchored VWAP starts wherever the story starts.
* Stamp bars at their close and align higher timeframes on **completed** bars only.
* A basis-point edge from a one-bar rule on random data is a leak, not an edge.
* Graded version: `labs/part05/week18_groups` (`session_vwap`, `anchored_vwap`, `volume_profile` with value area, `align_higher_tf`).